# Whole Dataset - Census Geocoder

Geocodes addresses from chunked `2_whole_properties_for_geocoding_standardized_*.csv` and outputs `3_whole_properties_census_geocoded_*.csv` (≤100MB each for GitHub).

**Features:** Resume support, checkpoint saving, Tie resolution. Run standardizer first.

In [1]:
import os
import io
import re
import glob
import time
import pandas as pd
import requests

# --- Config ---
DATA_LABEL = "whole"
DATA_DIR = "../../data/1_derived"
INPUT_PATTERN = os.path.join(DATA_DIR, f"2_{DATA_LABEL}_properties_for_geocoding_standardized_*.csv")
OUT_PREFIX = f"3_{DATA_LABEL}_properties_census_geocoded"
CENSUS_BATCH_URL = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
CENSUS_SINGLE_URL = "https://geocoding.geo.census.gov/geocoder/geographies/address"
CHUNK_SIZE = 500
MAX_ROWS_PER_FILE = 100_000

c:\Users\clint\Desktop\Geocoding_Office\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
def parse_address(addr):
    parts = [p.strip() for p in str(addr).split(',')]
    street = parts[0] if len(parts) > 0 else ''
    city = parts[1] if len(parts) > 1 else ''
    state_zip = parts[2].strip() if len(parts) > 2 else ''
    tokens = state_zip.split()
    state = tokens[0] if len(tokens) > 0 else ''
    zip_code = tokens[1] if len(tokens) > 1 else ''
    return street, city, state, zip_code

def geocode_batch(chunk_df, retries=5, base_sleep=3):
    csv_bytes = chunk_df.to_csv(index=False, header=False).encode("utf-8")
    for attempt in range(1, retries + 1):
        try:
            resp = requests.post(
                CENSUS_BATCH_URL,
                files={"addressFile": ("addresses.csv", csv_bytes, "text/csv")},
                data={"benchmark": "Public_AR_Current", "vintage": "Current_Current"},
                timeout=300,
            )
            resp.raise_for_status()
            body = resp.text.strip()
            if not body:
                raise ValueError("Empty response body from batch endpoint")
            return body
        except Exception as e:
            print(f"    Batch attempt {attempt} failed: {e}")
            if attempt < retries:
                time.sleep(base_sleep * attempt)
    return ""

def parse_census_response(raw_text):
    cols = ["id", "input_address", "match", "match_type", "matched_address", "coordinates",
            "tiger_line_id", "side", "state_fips", "county_fips", "census_tract", "census_block"]
    if not str(raw_text).strip():
        return pd.DataFrame(columns=cols)
    try:
        out = pd.read_csv(
            io.StringIO(raw_text),
            names=cols,
            dtype={"state_fips": str, "county_fips": str, "census_tract": str, "census_block": str},
        )
        out["id"] = pd.to_numeric(out["id"], errors="coerce")
        out = out.dropna(subset=["id"]).copy()
        out["id"] = out["id"].astype(int)
        return out
    except Exception:
        return pd.DataFrame(columns=cols)

def make_no_match_record(row_id):
    return {
        "id": int(row_id),
        "match": "No_Match",
        "match_type": "No_Match",
        "matched_address": "",
        "coordinates": "",
        "state_fips": "",
        "county_fips": "",
        "census_tract": "",
        "census_block": "",
    }

def geocode_single_fallback(row_id, street, city, state, zipcode, retries=3):
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(CENSUS_SINGLE_URL, params={
                "street": street, "city": city, "state": state, "zip": zipcode,
                "benchmark": "Public_AR_Current", "vintage": "Current_Current", "format": "json",
            }, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            matches = data.get("result", {}).get("addressMatches", [])
            if not matches:
                return [make_no_match_record(row_id)]

            out = []
            is_tie = len(matches) > 1
            for m in matches:
                coords = m.get("coordinates", {})
                blocks = (m.get("geographies", {}).get("2020 Census Blocks", []) or [{}])[0]
                out.append({
                    "id": int(row_id),
                    "match": "Tie" if is_tie else "Match",
                    "match_type": "Tie" if is_tie else "Exact",
                    "matched_address": m.get("matchedAddress", ""),
                    "coordinates": f"{coords.get('x', '')},{coords.get('y', '')}",
                    "state_fips": str(blocks.get("STATE", "")),
                    "county_fips": str(blocks.get("COUNTY", "")),
                    "census_tract": str(blocks.get("TRACT", "")),
                    "census_block": str(blocks.get("BLOCK", "")),
                })
            return out
        except Exception:
            if attempt < retries:
                time.sleep(2 * attempt)
    return [make_no_match_record(row_id)]

def resolve_tie_single(row_id, street, city, state, zipcode, retries=3):
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(CENSUS_SINGLE_URL, params={
                "street": street, "city": city, "state": state, "zip": zipcode,
                "benchmark": "Public_AR_Current", "vintage": "Current_Current", "format": "json",
            }, timeout=60)
            resp.raise_for_status()
            data = resp.json()
            matches = data.get("result", {}).get("addressMatches", [])
            if not matches:
                return []
            candidates = []
            for m in matches:
                coords = m.get("coordinates", {})
                blocks = (m.get("geographies", {}).get("2020 Census Blocks", []) or [{}])[0]
                candidates.append({
                    "id": row_id, "match": "Tie", "match_type": "Tie",
                    "matched_address": m.get("matchedAddress", ""),
                    "coordinates": f"{coords.get('x','')},{coords.get('y','')}",
                    "state_fips": str(blocks.get("STATE", "")), "county_fips": str(blocks.get("COUNTY", "")),
                    "census_tract": str(blocks.get("TRACT", "")), "census_block": str(blocks.get("BLOCK", "")),
                })
            return candidates
        except Exception:
            if attempt < retries:
                time.sleep(2)
    return []

In [3]:
def split_coords(coord_str):
    try:
        lon, lat = str(coord_str).split(",")
        return float(lon.strip()), float(lat.strip())
    except Exception:
        return None, None

def process_one_chunk(df_chunk, existing_matched_ids, out_path):
    """Geocode rows not in existing_matched_ids. If all done, load from out_path. Returns df_geo."""
    to_geocode = df_chunk[~df_chunk["PropertyId"].isin(existing_matched_ids)].copy()
    if to_geocode.empty:
        if os.path.exists(out_path):
            return pd.read_csv(out_path)
        existing_files = sorted(glob.glob(os.path.join(DATA_DIR, f"{OUT_PREFIX}_*.csv")))
        if existing_files:
            geo_parts = []
            for f in existing_files:
                ex = pd.read_csv(f)
                gc = [c for c in ["PropertyId", "match", "latitude", "longitude", "geoid"] if c in ex.columns]
                if gc:
                    geo_parts.append(ex[gc].drop_duplicates("PropertyId", keep="first"))
            if geo_parts:
                prev = pd.concat(geo_parts, ignore_index=True).drop_duplicates("PropertyId", keep="first")
                return df_chunk.merge(prev, on="PropertyId", how="left")
        return df_chunk

    parsed = to_geocode["full_address_std"].apply(parse_address)
    batch_input = pd.DataFrame(parsed.tolist(), columns=["street", "city", "state", "zip"])
    batch_input.insert(0, "id", to_geocode["PropertyId"].astype(int).values)

    results = []
    for i in range(0, len(batch_input), CHUNK_SIZE):
        chunk = batch_input.iloc[i:i+CHUNK_SIZE]
        raw = geocode_batch(chunk)
        parsed_chunk = parse_census_response(raw)

        chunk_ids = set(chunk["id"].astype(int).tolist())
        returned_ids = set(parsed_chunk["id"].astype(int).tolist()) if not parsed_chunk.empty else set()
        missing_ids = sorted(chunk_ids - returned_ids)

        if missing_ids:
            print(f"    Batch returned {len(returned_ids)}/{len(chunk_ids)} rows; fallback for {len(missing_ids)} rows")
            fallback = []
            missing_rows = chunk[chunk["id"].isin(missing_ids)]
            for _, r in missing_rows.iterrows():
                fallback.extend(geocode_single_fallback(r["id"], r["street"], r["city"], r["state"], r["zip"]))
                time.sleep(0.1)
            if fallback:
                parsed_chunk = pd.concat([parsed_chunk, pd.DataFrame(fallback)], ignore_index=True)

        results.append(parsed_chunk)
        time.sleep(0.5)

    geo_df = pd.concat(results, ignore_index=True) if results else pd.DataFrame()

    # Safety net: guarantee one output record per requested id.
    if not geo_df.empty:
        geo_df["id"] = pd.to_numeric(geo_df["id"], errors="coerce")
        geo_df = geo_df.dropna(subset=["id"]).copy()
        geo_df["id"] = geo_df["id"].astype(int)
    returned_all = set(geo_df["id"].tolist()) if not geo_df.empty else set()
    requested_all = set(batch_input["id"].astype(int).tolist())
    still_missing = sorted(requested_all - returned_all)
    if still_missing:
        geo_df = pd.concat([geo_df, pd.DataFrame([make_no_match_record(mid) for mid in still_missing])], ignore_index=True)

    tie_ids = geo_df.loc[geo_df["match"] == "Tie", "id"].unique()
    if len(tie_ids) > 0:
        tie_input = batch_input[batch_input["id"].isin(tie_ids)].set_index("id")
        resolved = []
        for tid in tie_ids:
            row = tie_input.loc[tid]
            resolved.extend(resolve_tie_single(tid, row["street"], row["city"], row["state"], row["zip"]))
            time.sleep(0.25)
        if resolved:
            geo_df = geo_df[geo_df["match"] != "Tie"]
            geo_df = pd.concat([geo_df, pd.DataFrame(resolved)], ignore_index=True)

    geo_df[["longitude", "latitude"]] = pd.DataFrame(geo_df["coordinates"].apply(split_coords).tolist(), index=geo_df.index)
    for c in ["state_fips", "county_fips", "census_tract", "census_block"]:
        if c in geo_df.columns:
            geo_df[c] = geo_df[c].fillna("").astype(str).str.zfill(2 if c == "state_fips" else 3 if c == "county_fips" else 6 if c == "census_tract" else 4)
    geo_df["geoid"] = geo_df["state_fips"].str.cat([geo_df["county_fips"], geo_df["census_tract"], geo_df["census_block"]], sep="")

    tie_mask = geo_df["match"] == "Tie"
    if tie_mask.any():
        tie_agg = geo_df[tie_mask].groupby("id").agg(
            tie_matched_addresses=("matched_address", lambda x: " | ".join(x.fillna("").astype(str))),
            tie_latitudes=("latitude", lambda x: " | ".join(x.dropna().astype(str))),
            tie_longitudes=("longitude", lambda x: " | ".join(x.dropna().astype(str))),
            tie_geoids=("geoid", lambda x: " | ".join(x.fillna("").astype(str))),
        ).reset_index()
        geo_df = geo_df.drop_duplicates(subset="id", keep="first")
        geo_df = geo_df.merge(tie_agg, on="id", how="left")
    else:
        geo_df = geo_df.drop_duplicates(subset="id", keep="first")
        geo_df["tie_matched_addresses"] = geo_df["tie_latitudes"] = geo_df["tie_longitudes"] = geo_df["tie_geoids"] = None

    geo_df["id"] = pd.to_numeric(geo_df["id"], errors="coerce")
    new_geo = geo_df[["id", "match", "matched_address", "latitude", "longitude", "state_fips", "county_fips", "census_tract", "census_block", "geoid", "tie_matched_addresses", "tie_latitudes", "tie_longitudes", "tie_geoids"]].rename(columns={"id": "PropertyId"})
    return df_chunk.merge(new_geo, on="PropertyId", how="left")

In [4]:
# Load existing geocoded PropertyIds for resume support
existing_files = sorted(glob.glob(os.path.join(DATA_DIR, f"{OUT_PREFIX}_*.csv")))
existing_matched_ids = set()
if existing_files:
    for ef in existing_files:
        ex = pd.read_csv(ef, usecols=["PropertyId", "match"])
        existing_matched_ids.update(ex.loc[ex["match"].isin(["Match", "Tie"]), "PropertyId"].dropna().astype(int).tolist())
    print(f"Resume: {len(existing_matched_ids):,} already matched from {len(existing_files)} file(s)")

geocode_cols = ["match", "matched_address", "latitude", "longitude", "state_fips", "county_fips", "census_tract", "census_block", "geoid", "tie_matched_addresses", "tie_latitudes", "tie_longitudes", "tie_geoids"]
input_files = sorted(glob.glob(INPUT_PATTERN))
print(f"Processing {len(input_files)} standardized chunk(s)\n")

Processing 11 standardized chunk(s)



In [5]:
# Process each standardized chunk, geocode, save. Supports resume.
for idx, inp_path in enumerate(input_files):
    chunk_num = inp_path.split("_")[-1].replace(".csv", "")
    out_name = f"{OUT_PREFIX}_{chunk_num}"
    out_path = os.path.join(DATA_DIR, f"{out_name}.csv")
    print(f"Chunk {idx+1}/{len(input_files)}: {os.path.basename(inp_path)}")
    df_full = pd.read_csv(inp_path)
    df_geo = process_one_chunk(df_full, existing_matched_ids, out_path)
    new_matched = df_geo.loc[df_geo["match"].isin(["Match", "Tie"]), "PropertyId"].dropna().astype(int).tolist()
    existing_matched_ids.update(new_matched)
    df_geo.to_csv(out_path, index=False)
    sz_mb = os.path.getsize(out_path) / 1e6
    n_match = (df_geo["match"] == "Match").sum()
    n_tie = (df_geo["match"] == "Tie").sum()
    print(f"  Saved → {out_name}.csv ({len(df_geo):,} rows, {sz_mb:.1f} MB) | Match: {n_match:,} | Tie: {n_tie}\n")

Chunk 1/11: 2_whole_properties_for_geocoding_standardized_001.csv
    Batch attempt 1 failed: HTTPSConnectionPool(host='geocoding.geo.census.gov', port=443): Read timed out. (read timeout=300)
  Saved → 3_whole_properties_census_geocoded_001.csv (100,000 rows, 38.2 MB) | Match: 91,150 | Tie: 886

Chunk 2/11: 2_whole_properties_for_geocoding_standardized_002.csv
    Batch attempt 1 failed: HTTPSConnectionPool(host='geocoding.geo.census.gov', port=443): Read timed out. (read timeout=300)
  Saved → 3_whole_properties_census_geocoded_002.csv (100,000 rows, 37.4 MB) | Match: 92,865 | Tie: 519

Chunk 3/11: 2_whole_properties_for_geocoding_standardized_003.csv
  Saved → 3_whole_properties_census_geocoded_003.csv (100,000 rows, 37.4 MB) | Match: 93,632 | Tie: 677

Chunk 4/11: 2_whole_properties_for_geocoding_standardized_004.csv
    Batch attempt 1 failed: HTTPSConnectionPool(host='geocoding.geo.census.gov', port=443): Read timed out. (read timeout=300)
  Saved → 3_whole_properties_census_geoc